In [4]:
# Central config — run this at the start of EVERY session
import os

CONFIG = {
    "OPENAI_API_KEY": os.environ.get("OPENAI_API_KEY", ""),
    "OPENAI_MODEL": "gpt-4o-mini",           # Use mini to save costs during dev
    "DATABASE_URL": os.environ.get("DATABASE_URL", "sqlite:////content/rag.db"),
    "EMBEDDING_MODEL": "BAAI/bge-base-en-v1.5",  # base = smaller/faster for Colab
    "EMBEDDING_DIMENSION": "768",                  # base model = 768 dims
    "RERANKER_MODEL": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "BM25_WEIGHT": "0.3",
    "VECTOR_WEIGHT": "0.7",
    "TOP_K_RETRIEVE": "20",
    "TOP_K_RERANK": "5",
    "RETRIEVAL_QUALITY_THRESHOLD": "0.5",
    "MIN_FEEDBACK_SAMPLES": "5",
    "LEARNING_RATE": "0.01",
    "PROJECT_DIR": "/content/drive/MyDrive/SelfImprovingRAG"
}

for key, value in CONFIG.items():
    os.environ[key] = str(value)

print("✅ Environment configured:")
for k, v in CONFIG.items():
    if "KEY" in k:
        print(f"   {k}: {'✅ set' if v else '❌ MISSING'}")
    else:
        print(f"   {k}: {v}")

✅ Environment configured:
   OPENAI_API_KEY: ❌ MISSING
   OPENAI_MODEL: gpt-4o-mini
   DATABASE_URL: sqlite:////content/rag.db
   EMBEDDING_MODEL: BAAI/bge-base-en-v1.5
   EMBEDDING_DIMENSION: 768
   RERANKER_MODEL: cross-encoder/ms-marco-MiniLM-L-6-v2
   BM25_WEIGHT: 0.3
   VECTOR_WEIGHT: 0.7
   TOP_K_RETRIEVE: 20
   TOP_K_RERANK: 5
   RETRIEVAL_QUALITY_THRESHOLD: 0.5
   MIN_FEEDBACK_SAMPLES: 5
   LEARNING_RATE: 0.01
   PROJECT_DIR: /content/drive/MyDrive/SelfImprovingRAG


In [5]:
# Paste and run the CONFIG cell from Notebook 1 first!
# Then continue here.

import hashlib
import re
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from pathlib import Path

print("✅ Imports ready")

✅ Imports ready


In [6]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    parent_chunk_id: Optional[str] = None
    chunk_index: int = 0
    total_chunks: int = 0

    def __post_init__(self):
        if not self.chunk_id:
            content_hash = hashlib.md5(
                f"{self.doc_id}:{self.chunk_index}:{self.content[:100]}".encode()
            ).hexdigest()
            self.chunk_id = f"chunk_{content_hash[:12]}"

    def _generate_id(self) -> str:
        content_hash = hashlib.md5(
            f"{self.doc_id}:{self.chunk_index}:{self.content[:100]}".encode()
        ).hexdigest()
        return f"chunk_{content_hash[:12]}"


@dataclass
class Document:
    doc_id: str
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    source: str = ""

    def __post_init__(self):
        if not self.doc_id:
            self.doc_id = hashlib.md5(self.content[:500].encode()).hexdigest()[:12]


print("✅ Chunk and Document classes defined")

✅ Chunk and Document classes defined


In [7]:
class SemanticChunker:
    """
    Hierarchical chunker.
    child_chunk_size: small chunks for retrieval
    parent_chunk_size: full chunks sent to LLM
    """
    def __init__(self, child_chunk_size=128, parent_chunk_size=512,
                 child_overlap=20, parent_overlap=50):
        self.child_chunk_size = child_chunk_size
        self.parent_chunk_size = parent_chunk_size
        self.child_overlap = child_overlap
        self.parent_overlap = parent_overlap

    def _split_into_sentences(self, text: str) -> List[str]:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sentences if s.strip() and len(s.strip()) > 10]

    def _create_chunks(self, sentences, chunk_size, overlap, doc_id, prefix):
        chunks = []
        current = []
        current_tokens = 0

        for sentence in sentences:
            token_est = len(sentence) // 4
            if current_tokens + token_est > chunk_size and current:
                content = " ".join(current)
                c = Chunk(chunk_id="", doc_id=doc_id, content=content,
                         chunk_index=len(chunks), metadata={"prefix": prefix})
                c.chunk_id = c._generate_id()
                chunks.append(c)
                current = current[-overlap:] + [sentence] if overlap else [sentence]
                current_tokens = sum(len(s)//4 for s in current)
            else:
                current.append(sentence)
                current_tokens += token_est

        if current:
            content = " ".join(current)
            c = Chunk(chunk_id="", doc_id=doc_id, content=content,
                     chunk_index=len(chunks), metadata={"prefix": prefix})
            c.chunk_id = c._generate_id()
            chunks.append(c)

        for chunk in chunks:
            chunk.total_chunks = len(chunks)
        return chunks

    def chunk_document(self, document: Document):
        sentences = self._split_into_sentences(document.content)
        parent_chunks = self._create_chunks(
            sentences, self.parent_chunk_size, self.parent_overlap,
            document.doc_id, "parent"
        )
        for chunk in parent_chunks:
            chunk.metadata.update(document.metadata)
            chunk.metadata["source"] = document.source

        child_chunks = []
        for parent in parent_chunks:
            parent_sents = self._split_into_sentences(parent.content)
            children = self._create_chunks(
                parent_sents, self.child_chunk_size, self.child_overlap,
                document.doc_id, "child"
            )
            for child in children:
                child.parent_chunk_id = parent.chunk_id
                child.metadata.update(parent.metadata)
            child_chunks.extend(children)

        return parent_chunks, child_chunks


print("✅ SemanticChunker defined")

✅ SemanticChunker defined


In [8]:
from google.colab import files
import os # Import os module
print("📎 Click 'Choose Files' to upload a PDF (or any .txt file)")
uploaded = files.upload()  # This opens a file picker

# Save uploaded files to our project dir
import shutil
for filename in uploaded.keys():
    dest = f"/content/drive/MyDrive/SelfImprovingRAG/data/raw/{filename}"
    os.makedirs(os.path.dirname(dest), exist_ok=True) # Create directory if it doesn't exist
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f"✅ Saved: {dest}")

📎 Click 'Choose Files' to upload a PDF (or any .txt file)


Saving Suresh_Data_resume1.pdf to Suresh_Data_resume1 (2).pdf
✅ Saved: /content/drive/MyDrive/SelfImprovingRAG/data/raw/Suresh_Data_resume1 (2).pdf
